In [ ]:
import pathlib as pl
from pprint import pprint

import pywatershed as pws

In [ ]:
output_dir = pl.Path("./11_cascading_flow/output")
domain_dir = pl.Path("../test_data/sagehen_5yr/")

In [ ]:
control_file = domain_dir / "sagehen_no_gw_cascades.control"
control = pws.Control.load_prms(control_file)
control.options["parameter_file"] = (
    domain_dir / control.options["parameter_file"][2:]
)
control.options["input_dir"] = domain_dir
control.options["imbalance_behavior"] = None
control.options["netcdf_output_dir"] = output_dir
control.options["netcdf_output_var_names"] = [
    *pws.PRMSRunoffCascadesNoDprst.get_variables(),
    *pws.PRMSSoilzoneCascadesNoDprst.get_variables(),
    *pws.PRMSGroundwaterNoDprst.get_variables(),
]

In [ ]:
parameters = pws.parameters.PrmsParameters.load(
    control.options["parameter_file"]
)

In [ ]:
# Derive the cascade parameters (hru_route_order, hru_down, ...) from the
# PRMS parameter file. PRMSRunoffCascadesNoDprst and
# PRMSSoilzoneCascadesNoDprst do this themselves when hru_route_order is
# missing, so this step is optional. Calling it explicitly is useful to
# inspect or save the derived parameters, or to preprocess once and share
# the result between the two cascade processes rather than have each
# derive it.
parameters = pws.utils.preprocess_cascades.preprocess_cascade_params(
    control, parameters
)

In [ ]:
dis_hru = pws.Parameters.from_netcdf(
    domain_dir / "parameters_dis_hru.nc", encoding=False
)

process_dict = {
    "solar": pws.PRMSSolarGeometry,
    "atmosphere": pws.PRMSAtmosphere,
    "canopy": pws.PRMSCanopy,
    "snow": pws.PRMSSnow,
    "runoff": pws.PRMSRunoffCascadesNoDprst,
    "soilzone": pws.PRMSSoilzoneCascadesNoDprst,
    "groundwater": pws.PRMSGroundwaterNoDprst,
}

model_dict = {
    "control": control,
    "dis_hru": dis_hru,
    "model_order": list(process_dict.keys()),
}

for kk, vv in process_dict.items():
    model_dict[kk] = {"class": vv, "parameters": parameters, "dis": "dis_hru"}

In [ ]:
pprint(model_dict, sort_dicts=False)

In [ ]:
model = pws.Model(model_dict)

In [ ]:
model.run(netcdf_dir=output_dir, finalize=True)

In [ ]:
import xarray as xr

# load_dataarray reads and closes the file (open_dataarray would keep it
# open, blocking a later rerun of the model from overwriting it)
horton_casc = xr.load_dataarray(output_dir / "hru_horton_cascflow.nc")
sz_casc = xr.load_dataarray(output_dir / "hru_sz_cascadeflow.nc")

In [ ]:
# The sagehen HRU polygons are the "HRU" layer of the Sagehen.gdb
# geodatabase in the pywatershed GIS files (pws.utils.gis_files.download()).
proc_plot = pws.analysis.process_plot.ProcessPlot(
    pws.utils.get_gis_dir("sagehen_mf6"),
    hru_shp_file_name="Sagehen.gdb",
    hru_layer="HRU",
    seg_shp_file_name=None,
)

# Time-mean Hortonian surface runoff cascading to downslope HRUs
proc_plot.plot_hru_var(
    var_name="hru_horton_cascflow",
    process=model.processes["runoff"],
    data=horton_casc.mean(dim="time"),
    data_units=horton_casc.attrs["units"],
    nhm_id=horton_casc["nhm_id"],
)

In [ ]:
# Time-mean soilzone (interflow and Dunnian) flow cascading downslope
proc_plot.plot_hru_var(
    var_name="hru_sz_cascadeflow",
    process=model.processes["soilzone"],
    data=sz_casc.mean(dim="time"),
    data_units=sz_casc.attrs["units"],
    nhm_id=sz_casc["nhm_id"],
)

In [ ]:
import panel as pn

pn.extension()

# Through time, one map per month with a play/pause "scrubber" widget.
# Hortonian cascade flow happens on a few melt or rain days, so the monthly
# maximum shows where and when; the mean would average it away. On this
# domain it amounts to two events in five years. The color scale is fixed
# across months so they compare. .embed() puts all frames in the output so
# playback runs in the browser.
horton_monthly = horton_casc.resample(time="MS").max()
horton_maps = proc_plot.plot_hru_var(
    var_name="hru_horton_cascflow",
    process=model.processes["runoff"],
    data=horton_monthly,
    data_units=horton_casc.attrs["units"],
    nhm_id=horton_casc["nhm_id"],
    clim=(0.0, float(horton_monthly.max())),
)
pn.panel(horton_maps, widget_type="scrubber", widget_location="bottom").embed()

In [ ]:
# Soilzone cascade flow (interflow and Dunnian) is sustained over weeks,
# so the monthly mean is the right summary here.
sz_monthly = sz_casc.resample(time="MS").mean()
sz_maps = proc_plot.plot_hru_var(
    var_name="hru_sz_cascadeflow",
    process=model.processes["soilzone"],
    data=sz_monthly,
    data_units=sz_casc.attrs["units"],
    nhm_id=sz_casc["nhm_id"],
    clim=(0.0, float(sz_monthly.max())),
)
pn.panel(sz_maps, widget_type="scrubber", widget_location="bottom").embed()